## LLM With Agentic Loop And Message History

In [ ]:
import os
import inspect
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, SecretStr, BaseModel, ValidationError, create_model
import json 
from functools import wraps
import numexpr as ne
from typing import Callable, Any, Dict

In [ ]:
class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file="../.env")
    groq_api_key: SecretStr
    openai_api_key: SecretStr

settings = AppSettings()
print(settings.groq_api_key)   


**********


### The expected structure from LLM
* City name along with weather information.
* Fahrenheit instead of Celsius.

# Langchain Style Tool Creation Decorator

In [ ]:
class FuncationMetadataTool:
    """Wraps a Python function with metadata for an LLM."""
    def __init__(self, func: Callable, name: str, description: str, args_schema: type[BaseModel]):
        self.func = func
        self.name = name
        self.description = description
        self.args_schema = args_schema

    def __call__(self, *args, **kwargs) -> Any:
        # Validates arguments against the Pydantic schema before execution
        validated_args = self.args_schema(**kwargs)
        return self.func(**validated_args.model_dump())

    def get_llm_schema(self) -> Dict[str, Any]:
        """Generates OpenAI-style tool definition schema."""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.args_schema.model_json_schema()
            }
        }

def tool(func: Callable) -> FuncationMetadataTool:
    """Decorator to transform a function into a CustomTool."""
    # Extract name and description
    name = func.__name__
    description = func.__doc__ or "No description provided."
    
    # Extract function signatures and type hints
    sig = inspect.signature(func)
    fields = {}
    
    for param_name, param in sig.parameters.items():
        if param_name == 'self':
            continue
        # Default to Any if no type hint is provided
        param_type = param.annotation if param.annotation != inspect.Parameter.empty else Any
        # Handle default values
        default_value = param.default if param.default != inspect.Parameter.empty else ...
        fields[param_name] = (param_type, default_value)
    
    # Dynamically create a Pydantic model for input validation
    schema_name = f"{name}"
    args_schema = create_model(schema_name, **fields)
    
    return FuncationMetadataTool(func, name, description, args_schema)


### Dummy Weather and Calculator Tool.

In [28]:
@tool
def get_weather_information(city: str):
    """
    Retrieve weather information for a supported city.
    Args:
        city (str): The name of the city.
    Returns:
        dict: A dictionary containing:
            - celsius (int): Temperature in degrees Celsius.
            - conditions (str): A brief description of the weather.
    """
    weather = {
        "tokyo": {"celsius": 22, "conditions": "partly cloudy"},
        "delhi": {"celsius": 34, "conditions": "clear skies"},
        "london": {"celsius": 15, "conditions": "light rain"},
    }
    return weather.get(city.lower())

@tool
def calculator(expression: str) -> str:
    """
    Calculates mathematical expressions using numexpr.
    
    Args:
        expression: A string mathematical expression (e.g., "5.6 * (5 + 10.5)").
        
    Returns:
        The result of the calculation as a string.
    """
    try:
        result = ne.evaluate(expression)
        return f"The result of '{expression}' is {result}"
    except Exception as e:
        return f"Error evaluating expression: {e}"

TOOL_SCHEMAS = [get_weather_information.get_llm_schema(), calculator.get_llm_schema()]
print(json.dumps(TOOL_SCHEMAS, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "get_weather_information",
      "description": "\nRetrieve weather information for a supported city.\nArgs:\n    city (str): The name of the city.\nReturns:\n    dict: A dictionary containing:\n        - celsius (int): Temperature in degrees Celsius.\n        - conditions (str): A brief description of the weather.\n",
      "parameters": {
        "properties": {
          "city": {
            "title": "City",
            "type": "string"
          }
        },
        "required": [
          "city"
        ],
        "title": "get_weather_information",
        "type": "object"
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "calculator",
      "description": "\nCalculates mathematical expressions using numexpr.\n\nArgs:\n    expression: A string mathematical expression (e.g., \"5.6 * (5 + 10.5)\").\n\nReturns:\n    The result of the calculation as a string.\n",
      "parameters": {
        

In [22]:
TOOLS_BY_NAME = {"get_weather_information": get_weather_information, 'calculator': calculator}

### LLM with Tools Enabled
* Sends the question plus the tool schema in one call. 
* The reply may contain content or tool_calls list instead suggested by LLM.

### The agentic loop
* In each iteration, call the model with the attached tool.
* If it replies with tool_calls, execute every one of them in same iteration and feed the results back to LLM.
* If it replies with plain content instead, that is the final answer and the loop stops. 
* max_turns is a safety limit so a confused model can't loop forever.

In [77]:
def get_client_and_model():
    from openai import OpenAI
    return OpenAI(api_key=settings.openai_api_key.get_secret_value()), "gpt-4o-mini"


def run_agent(messages: list, max_turns: int = 4) -> str:
    client, model = get_client_and_model()
    
    for i in range(max_turns):
        # print(f'Message {i}: ', messages)
        response = client.chat.completions.create(
            model=model, max_tokens=300, messages=messages, tools=TOOL_SCHEMAS
        )
        message = response.choices[0].message
        print(f"LLM Response on iter {1}: ", message)


        if not message.tool_calls:
            messages.append({"role": "assistant", "content": message.content})
            return message.content, messages
        
        for call in message.tool_calls:
            messages.append(
                {
                    "role": "assistant",
                    "content": message.content,
                    "tool_calls": [
                        {
                            "id": call.id,
                            "type": "function",
                            "function": {"name": call.function.name, "arguments": call.function.arguments},
                        }
                    ],
                }
            )
            print(f"Iter {i} Called Tool: {call.function.name} with params {call.function.arguments}")
            arguments = json.loads(call.function.arguments)
            tool_function = TOOLS_BY_NAME[call.function.name]
            result = tool_function(**arguments)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})


In [78]:
user_input = 'Choose available tools when required and answer below question\n what is the current weather in capital of japan And solve eqation (8+4)/6+4'
message = [{"role": "user", "content": user_input}]
answer, message_history = run_agent(message)

LLM Response on iter 1:  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_d0VwgziwBeTJOvuE8J6wl06K', function=Function(arguments='{"city": "Tokyo"}', name='get_weather_information'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_oc6LkkJ2pAyA6D9UJYzs36hL', function=Function(arguments='{"expression": "(8+4)/6+4"}', name='calculator'), type='function')])
Iter 0 Called Tool: get_weather_information with params {"city": "Tokyo"}
Iter 0 Called Tool: calculator with params {"expression": "(8+4)/6+4"}
LLM Response on iter 1:  ChatCompletionMessage(content='The current weather in Tokyo, the capital of Japan, is 22°C and partly cloudy. \n\nAdditionally, the result of the equation \\((8+4)/6 + 4\\) is 6.0.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


## Final output from Agentic Loop

In [81]:
print(answer)

The current weather in Tokyo, the capital of Japan, is 22°C and partly cloudy. 

Additionally, the result of the equation \((8+4)/6 + 4\) is 6.0.


## Message history generated in Agentic Loop

In [82]:
message_history

[{'role': 'user',
  'content': 'Choose available tools when required and answer below question\n what is the current weather in capital of japan And solve eqation (8+4)/6+4'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'call_d0VwgziwBeTJOvuE8J6wl06K',
    'type': 'function',
    'function': {'name': 'get_weather_information',
     'arguments': '{"city": "Tokyo"}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_d0VwgziwBeTJOvuE8J6wl06K',
  'content': "{'celsius': 22, 'conditions': 'partly cloudy'}"},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'call_oc6LkkJ2pAyA6D9UJYzs36hL',
    'type': 'function',
    'function': {'name': 'calculator',
     'arguments': '{"expression": "(8+4)/6+4"}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_oc6LkkJ2pAyA6D9UJYzs36hL',
  'content': "The result of '(8+4)/6+4' is 6.0"},
 {'role': 'assistant',
  'content': 'The current weather in Tokyo, the capital of Japan, is 22°C and partly cloudy. \n\nAdditionally, the r